# Demo 5: Building Agents

This demo covers agent anatomy (the reason-act-observe loop), building a single agent with UC function tools, the @function_tool decorator, and how a single agent orchestrates its own tools.

In [0]:
%sql
-- SETUP: Create catalog, schema, sample data, and UC functions
-- for building and testing a single agent.

CREATE CATALOG IF NOT EXISTS module5a_demo5;
CREATE SCHEMA IF NOT EXISTS module5a_demo5.agent_tools;

-- Sample support tickets table
CREATE OR REPLACE TABLE module5a_demo5.agent_tools.support_tickets (
  ticket_id    STRING NOT NULL,
  customer_id  STRING NOT NULL,
  category     STRING,
  priority     STRING,
  subject      STRING,
  description  STRING,
  status       STRING,
  created_at   TIMESTAMP
);

INSERT INTO module5a_demo5.agent_tools.support_tickets VALUES
('TKT-001', 'CUST-001', 'billing',   'high',   'Overcharged on order ORD-001', 'I was charged $249.99 but the product was on sale for $199.99', 'open', '2026-09-25 08:00:00'),
('TKT-002', 'CUST-002', 'shipping', 'medium', 'Order ORD-003 not delivered',   'My order from Sept 20 has not arrived yet',                     'open', '2026-09-25 09:30:00'),
('TKT-003', 'CUST-003', 'technical','high',   'Cannot access my account',     'I keep getting a 403 error when trying to log in',             'open', '2026-09-25 10:15:00'),
('TKT-004', 'CUST-001', 'general',   'low',    'Product recommendation',      'Can you recommend accessories for my recent purchase?',        'open', '2026-09-25 11:00:00'),
('TKT-005', 'CUST-002', 'billing',   'medium', 'Discount not applied',        'My premium discount was not applied to order ORD-008',         'open', '2026-09-25 11:45:00');

-- UC functions as agent tools
CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.get_ticket(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the subject and description of a support ticket'
RETURN (SELECT max(concat_ws(' | ', subject, description))
       FROM module5a_demo5.agent_tools.support_tickets
       WHERE ticket_id = get_ticket.ticket_id);

CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.count_tickets_by_category(cat STRING)
RETURNS INT
COMMENT 'Returns the number of tickets in a given category'
RETURN SELECT count(*) FROM module5a_demo5.agent_tools.support_tickets
       WHERE category = count_tickets_by_category.cat;

CREATE OR REPLACE FUNCTION module5a_demo5.agent_tools.get_priority(ticket_id STRING)
RETURNS STRING
COMMENT 'Returns the priority of a support ticket'
RETURN (SELECT max(priority) FROM module5a_demo5.agent_tools.support_tickets
       WHERE ticket_id = get_priority.ticket_id);

SELECT * FROM module5a_demo5.agent_tools.support_tickets ORDER BY ticket_id;

ticket_id,customer_id,category,priority,subject,description,status,created_at
TKT-001,CUST-001,billing,high,Overcharged on order ORD-001,I was charged .99 but the product was on sale for .99,open,2026-09-25T08:00:00.000Z
TKT-002,CUST-002,shipping,medium,Order ORD-003 not delivered,My order from Sept 20 has not arrived yet,open,2026-09-25T09:30:00.000Z
TKT-003,CUST-003,technical,high,Cannot access my account,I keep getting a 403 error when trying to log in,open,2026-09-25T10:15:00.000Z
TKT-004,CUST-001,general,low,Product recommendation,Can you recommend accessories for my recent purchase?,open,2026-09-25T11:00:00.000Z
TKT-005,CUST-002,billing,medium,Discount not applied,My premium discount was not applied to order ORD-008,open,2026-09-25T11:45:00.000Z


## Agent Anatomy: The Reason-Act-Observe Loop

### Concepts
An AI agent is not just a single LLM call. It follows a loop:

1. **Reason**: The LLM analyzes the user's request and decides what to do next.
   - "The user wants to know about TKT-001. I should call the get_ticket tool."

2. **Act**: The agent executes a tool (UC function, API call, SQL query).
   - Calls `get_ticket('TKT-001')` -> "Overcharged on order ORD-001 | I was charged..."

3. **Observe**: The agent sees the tool's output and incorporates it.
   - "The ticket is about a billing overcharge. The priority is high."

4. **Respond**: The agent synthesizes a natural language response.
   - "Ticket TKT-001 is a high-priority billing issue about an overcharge on order ORD-001."

**Key components of an agent**:
* **System prompt**: Defines the agent's role, capabilities, and constraints
* **Tools**: Functions the agent can call (UC functions, APIs, code)
* **Reasoning loop**: The reason-act-observe cycle that continues until the agent has enough information to respond

> A single LLM call is a question-answer pair. An agent is a **sequence of decisions** that may involve multiple tool calls before responding.

In [0]:
# Agent Anatomy Demo: The reason-act-observe loop
# Show each step of the loop explicitly for a single ticket query.

user_question = "What is the status of ticket TKT-001?"

print("=== Agent Reason-Act-Observe Loop ===")
print()
print(f"USER: {user_question}")
print()

# Step 1: REASON
print("--- STEP 1: REASON ---")
reasoning = spark.sql(f"""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'You are a support agent. A user asks: "{user_question}". You have these tools available: get_ticket(ticket_id), get_priority(ticket_id), count_tickets_by_category(category). Which tool should you call first? Respond with just the tool name and argument.',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
  ) AS reasoning
""").collect()[0][0]
print(f"Agent thinks: {reasoning.strip()}")
print()

# Step 2: ACT (call get_ticket)
print("--- STEP 2: ACT (call get_ticket) ---")
ticket_info = spark.sql("SELECT module5a_demo5.agent_tools.get_ticket('TKT-001')").collect()[0][0]
print(f"Tool output: {ticket_info}")
print()

# Step 3: ACT again (call get_priority)
print("--- STEP 2b: ACT (call get_priority) ---")
priority = spark.sql("SELECT module5a_demo5.agent_tools.get_priority('TKT-001')").collect()[0][0]
print(f"Tool output: {priority}")
print()

# Step 4: OBSERVE
print("--- STEP 3: OBSERVE ---")
print(f"Agent now knows: Ticket TKT-001 is about '{ticket_info[:50]}...' with priority '{priority}'")
print()

# Step 5: RESPOND
print("--- STEP 4: RESPOND ---")
response = spark.sql(f"""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'You are a support agent. Based on this information, respond to the user naturally. Ticket info: "{ticket_info}". Priority: "{priority}". User question: "{user_question}"',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 100)
  ) AS response
""").collect()[0][0]
print(f"Agent responds: {response.strip()}")

=== Agent Reason-Act-Observe Loop ===

USER: What is the status of ticket TKT-001?

--- STEP 1: REASON ---
Agent thinks: get_ticket(TKT-001)

--- STEP 2: ACT (call get_ticket) ---
Tool output: Overcharged on order ORD-001 | I was charged .99 but the product was on sale for .99

--- STEP 2b: ACT (call get_priority) ---
Tool output: high

--- STEP 3: OBSERVE ---
Agent now knows: Ticket TKT-001 is about 'Overcharged on order ORD-001 | I was charged .99 b...' with priority 'high'

--- STEP 4: RESPOND ---
Agent responds: I've located your ticket TKT-001 regarding the overcharge on order ORD-001. I apologize for the inconvenience you've experienced. I'm happy to inform you that I'm looking into this matter urgently, as it's been marked as high priority. 

To provide a brief update, I've reviewed your case, and it appears there might have been a misunderstanding with the pricing of the product, which was supposed to be on sale for $0.99. You were indeed charged $0


## 5.1 : Single-Agent vs. Multi-Agent

### When to use a single agent
A single agent is the right choice when:
* **Few tools** (< 8): The LLM can manage all tools effectively
* **One domain**: All questions fall within one area of expertise
* **Short instructions**: The system prompt fits comfortably in context
* **Simple flow**: The reason-act-observe loop doesn't need delegation

### When to split into multi-agent
* **Many tools** (8-10+): The LLM loses track of which tool to use
* **Multiple domains**: Billing, shipping, technical need different expertise
* **Long instructions**: System prompt exceeds ~2000 tokens
* **Team ownership**: Different teams own different domains

> This demo focuses on building a strong single agent. Demo 6 covers multi-agent orchestration.

## Building a Single Agent

### Concepts
A production single agent has three key pieces:

1. **System Prompt**: Defines the agent's persona, rules, and available tools
   - "You are a customer support agent. You can look up tickets, check priorities, and count tickets by category."

2. **Tool Definitions**: UC functions the agent can call
   - Each tool has a name, description, and parameter schema
   - The LLM uses the description to decide which tool to call

3. **Agent Loop**: The reason-act-observe cycle
   - The agent may call multiple tools in sequence before responding
   - Each tool result feeds back into the reasoning for the next step

> On Databricks, UC functions provide governed, auditable tools. The agent discovers them through their UC descriptions.

In [0]:
# Building a Single Agent: Complete agent with system prompt + tools + reasoning loop
# The agent handles different question types by choosing the right tools.

print("=== Single Agent: Handling Multiple Question Types ===")
print()

questions = [
    ("What is ticket TKT-003 about?", "TKT-003", "get_ticket"),
    ("How many billing tickets do we have?", "billing", "count_tickets_by_category"),
    ("What is the priority of TKT-002?", "TKT-002", "get_priority")
]

for question, arg, expected_tool in questions:
    print(f"USER: {question}")
    
    # Step 1: REASON - Agent decides which tool to call
    decision = spark.sql(f"""
      SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        'You are a support agent with these tools: get_ticket(ticket_id), get_priority(ticket_id), count_tickets_by_category(category). User asks: "{question}". Which tool should you call? Respond with just the tool name.',
        modelParameters => named_struct('temperature', 0.0, 'max_tokens', 30)
      ) AS decision
    """).collect()[0][0]
    print(f"  REASON -> Agent decides to call: {decision.strip()}")
    
    # Step 2: ACT - Execute the tool
    if 'get_ticket' in decision.lower():
        result = spark.sql(f"SELECT module5a_demo5.agent_tools.get_ticket('{arg}')").collect()[0][0]
    elif 'count' in decision.lower():
        result = str(spark.sql(f"SELECT module5a_demo5.agent_tools.count_tickets_by_category('{arg}')").collect()[0][0])
    elif 'priority' in decision.lower():
        result = spark.sql(f"SELECT module5a_demo5.agent_tools.get_priority('{arg}')").collect()[0][0]
    else:
        result = "Unknown tool"
    print(f"  ACT   -> Tool result: {str(result)[:70]}")
    
    # Step 3: OBSERVE + RESPOND
    response = spark.sql(f"""
      SELECT ai_query(
        'databricks-meta-llama-3-3-70b-instruct',
        'You are a support agent. Tool result: {str(result)[:200]}. Respond to: "{question}"',
        modelParameters => named_struct('temperature', 0.0, 'max_tokens', 80)
      ) AS response
    """).collect()[0][0]
    print(f"  RESPOND -> {response.strip()}")
    print()

=== Single Agent: Handling Multiple Question Types ===

USER: What is ticket TKT-003 about?
  REASON -> Agent decides to call: get_ticket
  ACT   -> Tool result: Cannot access my account | I keep getting a 403 error when trying to l
  RESPOND -> Ticket TKT-003 is related to an issue where the customer is unable to access their account due to a 403 error when attempting to log in. The error is preventing them from accessing their account, and they are seeking assistance to resolve the issue and regain access. Would you like me to provide more details or an update on the current status of the ticket?

USER: How many billing tickets do we have?
  REASON -> Agent decides to call: count_tickets_by_category
  ACT   -> Tool result: 2
  RESPOND -> We currently have 2 open billing tickets that require attention. Would you like me to provide more details about these tickets or assist with resolving them?

USER: What is the priority of TKT-002?
  REASON -> Agent decides to call: get_priority
  AC

## 5.4 : @function_tool Decorator

### Concepts
The `@function_tool` decorator converts a regular Python function into an agent tool:

* **What it does**: Takes a Python function with type hints and a docstring, and generates a tool schema that the LLM can understand.
* **Type hints** become parameter types in the tool schema.
* **Docstring** becomes the tool description for the LLM.
* **Return type** becomes the tool's output type.

This is the fastest way to create custom tools for agents - just write a function, decorate it, and the agent can call it.

> On Databricks, UC functions are the governed equivalent. @function_tool is for rapid prototyping; UC functions are for production.

In [0]:
# 5.4 Demo: @function_tool pattern - turning Python functions into agent tools
# The decorator converts type hints + docstring into a tool schema.
# We simulate the pattern here (the actual decorator requires the OpenAI Agents SDK).

import json

# Simulate the @function_tool decorator
def function_tool(func):
    """Simulate the @function_tool decorator that converts a function into a tool."""
    tool_schema = {
        "name": func.__name__,
        "description": func.__doc__.strip() if func.__doc__ else "",
        "parameters": {}
    }
    hints = func.__annotations__
    for param, hint in hints.items():
        if param != "return":
            tool_schema["parameters"][param] = hint.__name__ if hasattr(hint, '__name__') else str(hint)
    func.tool_schema = tool_schema
    return func

# Define tools using the decorator pattern
@function_tool
def get_ticket_status(ticket_id: str) -> str:
    """Returns the current status of a support ticket."""
    result = spark.sql(f"SELECT status FROM module5a_demo5.agent_tools.support_tickets WHERE ticket_id = '{ticket_id}'").collect()[0][0]
    return result

@function_tool
def count_category_tickets(category: str) -> int:
    """Returns the number of open tickets in a given category."""
    return spark.sql(f"SELECT module5a_demo5.agent_tools.count_tickets_by_category('{category}')").collect()[0][0]

@function_tool
def escalate_ticket(ticket_id: str, reason: str) -> str:
    """Escalates a ticket to a higher priority level."""
    return f"Ticket {ticket_id} escalated. Reason: {reason}"

# Show the auto-generated tool schemas
print("=== @function_tool: Auto-Generated Tool Schemas ===")
print()
for tool in [get_ticket_status, count_category_tickets, escalate_ticket]:
    schema = tool.tool_schema
    print(f"Tool: {schema['name']}")
    print(f"  Description: {schema['description']}")
    print(f"  Parameters: {schema['parameters']}")
    print()

# Call the tools (the agent would do this automatically)
print("=== Tool Execution ===")
print(f"  get_ticket_status('TKT-001') -> '{get_ticket_status('TKT-001')}'")
print(f"  count_category_tickets('billing') -> {count_category_tickets('billing')}")
print(f"  escalate_ticket('TKT-003', 'system outage') -> {escalate_ticket('TKT-003', 'system outage')}")

=== @function_tool: Auto-Generated Tool Schemas ===

Tool: get_ticket_status
  Description: Returns the current status of a support ticket.
  Parameters: {'ticket_id': 'str'}

Tool: count_category_tickets
  Description: Returns the number of open tickets in a given category.
  Parameters: {'category': 'str'}

Tool: escalate_ticket
  Description: Escalates a ticket to a higher priority level.
  Parameters: {'ticket_id': 'str', 'reason': 'str'}

=== Tool Execution ===
  get_ticket_status('TKT-001') -> 'open'
  count_category_tickets('billing') -> 2
  escalate_ticket('TKT-003', 'system outage') -> Ticket TKT-003 escalated. Reason: system outage


## 5.3 : Single-Agent Tool Orchestration

### Concepts
Even a single agent needs to orchestrate its tools:

1. **Sequential calls**: Call tool A, see the result, then call tool B.
   - "Get ticket TKT-001" -> see it's billing -> "Count billing tickets"

2. **Parallel calls**: Call multiple tools at once when independent.
   - "Get ticket TKT-001" + "Get priority TKT-001" at the same time

3. **Conditional calls**: Skip tools based on earlier results.
   - If the ticket is closed, don't check priority.

**LLM-driven vs. code-driven** (applies to single agents too):
* **LLM-driven**: The model decides which tool to call next based on context.
* **Code-driven**: The developer hard-codes the tool sequence.
* **Hybrid**: Code defines the overall flow; the LLM fills in details.

> In a single agent, the LLM is both the reasoner and the orchestrator. In multi-agent, a supervisor handles orchestration.

In [0]:
# 5.3 Demo: Single-agent tool orchestration
# Show how a single agent chains tool calls to answer a complex question.

print("=== Single Agent: Chained Tool Calls ===")
print()
print("User: 'Tell me about TKT-001 and how it compares to other billing tickets'")
print()

# Step 1: Get the ticket
print("--- Call 1: get_ticket('TKT-001') ---")
ticket = spark.sql("SELECT module5a_demo5.agent_tools.get_ticket('TKT-001')").collect()[0][0]
print(f"  Result: {ticket}")
print()

# Step 2: Get priority (independent of step 1, could be parallel)
print("--- Call 2: get_priority('TKT-001') ---")
priority = spark.sql("SELECT module5a_demo5.agent_tools.get_priority('TKT-001')").collect()[0][0]
print(f"  Result: {priority}")
print()

# Step 3: Count billing tickets (depends on knowing TKT-001 is billing)
print("--- Call 3: count_tickets_by_category('billing') ---")
billing_count = spark.sql("SELECT module5a_demo5.agent_tools.count_tickets_by_category('billing')").collect()[0][0]
print(f"  Result: {billing_count} billing tickets")
print()

# Step 4: Synthesize
print("--- Agent Synthesizes Response ---")
synthesis = spark.sql(f"""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'You are a support agent. Synthesize a response using these facts: Ticket TKT-001 details: {ticket}. Priority: {priority}. Total billing tickets: {billing_count}. User asked: Tell me about TKT-001 and how it compares to other billing tickets.',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 120)
  ) AS response
""").collect()[0][0]
print(f"  {synthesis.strip()}")
print()

print("=== Orchestration Summary ===")
print("The agent made 3 tool calls in sequence:")
print("  1. get_ticket -> learned the ticket is about billing")
print("  2. get_priority -> learned it's high priority")
print("  3. count_tickets_by_category -> learned there are 2 billing tickets")
print("Then synthesized a complete response from all 3 results.")

=== Single Agent: Chained Tool Calls ===

User: 'Tell me about TKT-001 and how it compares to other billing tickets'

--- Call 1: get_ticket('TKT-001') ---
  Result: Overcharged on order ORD-001 | I was charged .99 but the product was on sale for .99

--- Call 2: get_priority('TKT-001') ---
  Result: high

--- Call 3: count_tickets_by_category('billing') ---
  Result: 2 billing tickets

--- Agent Synthesizes Response ---
  I'd be happy to help you with your inquiry about Ticket TKT-001. According to our records, Ticket TKT-001 is related to an issue with your order ORD-001, where you were overcharged $0.99, despite the product being on sale for $0.99. This ticket has been marked as high priority, and our team is working to resolve the issue as soon as possible.

In terms of comparison, Ticket TKT-001 is one of two total billing tickets currently open. This suggests that billing issues are relatively rare, and we're committed to addressing them

=== Orchestration Summary ===
The agent m

## Learning Conclusion

### What we demonstrated

| Topic | What was demoed | Key Takeaway |
|---|---|---|
| Anatomy | Reason-act-observe loop | An agent is a sequence of decisions, not a single LLM call |
| 5.1 | Single vs. multi-agent decision | Start with single agent; split when tools >8 or domains differ |
| Building | Full single agent with system prompt + UC tools | System prompt + UC functions + reasoning loop = a working agent |
| 5.4 | @function_tool decorator | Type hints + docstring -> auto-generated tool schema |
| 5.3 | Single-agent tool orchestration | Agent chains tool calls: get ticket -> get priority -> count -> respond |

### Key principles
* **An agent is a loop, not a call**: Reason -> Act -> Observe -> Respond, repeating as needed.
* **UC functions are governed tools**: Register in Unity Catalog for production agents.
* **@function_tool for prototyping**: Quick tool creation; migrate to UC functions for governance.
* **Single agents can handle complex tasks**: Sequential and conditional tool calls within one agent.

In [0]:
# CLEANUP: Drop all resources created in this demo
print("Dropping UC functions...")
for fn in ['get_ticket', 'count_tickets_by_category', 'get_priority']:
    spark.sql(f"DROP FUNCTION IF EXISTS module5a_demo5.agent_tools.{fn}")
print("  Functions dropped")

print("Dropping schema and catalog...")
spark.sql("DROP SCHEMA IF EXISTS module5a_demo5.agent_tools CASCADE")
spark.sql("DROP CATALOG IF EXISTS module5a_demo5 CASCADE")
print("  Schema and catalog dropped")

print("\nCleanup complete!")